In [ ]:
%%sql -r dataframe_1
USE DATABASE ECOMMERCE_ANALYTICS;
USE SCHEMA MART;

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mticker
from scipy import stats
from snowflake.snowpark.context import get_active_session

In [ ]:
session = get_active_session()
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

## 1. Data Overview

In [ ]:
df_overview = session.sql("""
    SELECT
        COUNT(*)                                    AS total_events,
        COUNT(DISTINCT USER_ID)                     AS unique_users,
        COUNT(DISTINCT USER_SESSION)                AS unique_sessions,
        COUNT(DISTINCT PRODUCT_ID)                  AS unique_products,
        SUM(IS_PURCHASE)                            AS total_purchases,
        ROUND(SUM(CASE WHEN IS_PURCHASE = 1 
            THEN PRICE END), 2)                     AS total_revenue,
        ROUND(AVG(CASE WHEN IS_PURCHASE = 1 
            THEN PRICE END), 2)                     AS avg_order_value,
        MIN(EVENT_DATE)                             AS date_from,
        MAX(EVENT_DATE)                             AS date_to
    FROM FCT_EVENTS
""").to_pandas()

df_overview

### Key Findings:
- 3.7M unique users generated 67.5M events in November 2019
- Average user had 18 interactions during the month
- Average order value is $300 suggesting high ticket electronics store
- 916K purchases generated $275M in total revenue
- Each user on average had 3.7 sessions during the month

## 2. Funnel Analysis
Analysing the customer journey from product view to purchase.

In [ ]:
df_funnel = session.sql("""
    SELECT
        SUM(IS_VIEW)                                            AS views,
        SUM(IS_CART)                                            AS carts,
        SUM(IS_PURCHASE)                                        AS purchases,
        ROUND(SUM(IS_CART) / SUM(IS_VIEW) * 100, 2)            AS view_to_cart_rate,
        ROUND(SUM(IS_PURCHASE) / SUM(IS_CART) * 100, 2)        AS cart_to_purchase_rate,
        ROUND(SUM(IS_PURCHASE) / SUM(IS_VIEW) * 100, 2)        AS overall_conversion_rate
    FROM FCT_EVENTS
""").to_pandas()

df_funnel

In [ ]:
stages = ['Views', 'Carts', 'Purchases']
values = [df_funnel['VIEWS'][0], df_funnel['CARTS'][0], df_funnel['PURCHASES'][0]]

plt.bar(stages, values, color=['steelblue', 'orange', 'green'])
plt.title('Customer Purchase Funnel - November 2019')
plt.ylabel('Number of Events')
plt.tight_layout()
plt.show()

### Key Findings:
- Only 4.77% of views convert to cart
- 30.27% of carts convert to purchase
- Overall conversion rate is 1.44%
- Main opportunity: improve view to cart rate

## 3. Revenue Analysis
Identifying which categories and brands drive the most revenue.

In [ ]:
df_category = session.sql("""
    SELECT
        p.CATEGORY_L1,
        ROUND(SUM(f.PRICE), 2)          AS total_revenue,
        COUNT(*)                         AS total_purchases,
        ROUND(AVG(f.PRICE), 2)          AS avg_order_value
    FROM FCT_EVENTS f
    JOIN DIM_PRODUCTS p
        ON f.PRODUCT_ID = p.PRODUCT_ID
    WHERE f.IS_PURCHASE = 1
    GROUP BY p.CATEGORY_L1
    ORDER BY total_revenue DESC
    LIMIT 10
""").to_pandas()

df_category

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Revenue by category
axes[0].barh(df_category['CATEGORY_L1'], 
             df_category['TOTAL_REVENUE'],
             color='steelblue')
axes[0].set_title('Total Revenue by Category')
axes[0].set_xlabel('Revenue ($)')
axes[0].invert_yaxis()

# AOV by category
axes[1].barh(df_category['CATEGORY_L1'],
             df_category['AVG_ORDER_VALUE'],
             color='orange')
axes[1].set_title('Average Order Value by Category')
axes[1].set_xlabel('AOV ($)')
axes[1].invert_yaxis()

plt.suptitle('Revenue Analysis by Category', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Key Findings:
- Electronics dominates with $205M revenue (74.6% of total)
- Electronics and computers have highest AOV (~$400+)
- Unknown category accounts for $29M suggesting data quality impact
- Apparel has lowest AOV at $83 suggesting budget purchases

## 4. Top Brands Analysis
Identifying which brands drive the most revenue and purchases.

In [ ]:
df_brands = session.sql("""
    SELECT
        p.BRAND,
        ROUND(SUM(f.PRICE), 2)      AS total_revenue,
        COUNT(*)                     AS total_purchases,
        ROUND(AVG(f.PRICE), 2)      AS avg_order_value
    FROM FCT_EVENTS f
    JOIN DIM_PRODUCTS p
        ON f.PRODUCT_ID = p.PRODUCT_ID
    WHERE f.IS_PURCHASE = 1
    AND p.BRAND != 'unknown'
    GROUP BY p.BRAND
    ORDER BY total_revenue DESC
    LIMIT 10
""").to_pandas()

df_brands

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(df_brands['BRAND'],
             df_brands['TOTAL_REVENUE'],
             color='steelblue')
axes[0].set_title('Total Revenue by Brand')
axes[0].set_xlabel('Revenue ($)')
axes[0].invert_yaxis()

axes[1].barh(df_brands['BRAND'],
             df_brands['AVG_ORDER_VALUE'],
             color='orange')
axes[1].set_title('Average Order Value by Brand')
axes[1].set_xlabel('AOV ($)')
axes[1].invert_yaxis()

plt.suptitle('Top 10 Brands by Revenue - November 2019',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Key Findings:
- Apple leads revenue at \$127M with the highest AOV of \$768
- Samsung sells more units (200K) but at a lower AOV of \$274
- Two clear strategies: premium (Apple, Acer) vs volume (Samsung, Xiaomi)
- Top 2 brands account for 66% of total revenue

---
## 5. Time Analysis
Understanding when customers are most active and when purchases peak.

In [ ]:
df_hourly = session.sql("""
    SELECT
        EVENT_HOUR,
        COUNT(*)                    AS total_purchases,
        ROUND(SUM(PRICE), 2)        AS total_revenue
    FROM FCT_EVENTS
    WHERE IS_PURCHASE = 1
    GROUP BY EVENT_HOUR
    ORDER BY EVENT_HOUR
""").to_pandas()

df_hourly

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].plot(df_hourly['EVENT_HOUR'],
             df_hourly['TOTAL_PURCHASES'],
             color='steelblue', marker='o')
axes[0].set_title('Purchases by Hour')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Total Purchases')
axes[0].set_xticks(range(0, 24))

axes[1].plot(df_hourly['EVENT_HOUR'],
             df_hourly['TOTAL_REVENUE'],
             color='orange', marker='o')
axes[1].set_title('Revenue by Hour')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Revenue ($)')
axes[1].set_xticks(range(0, 24))

plt.suptitle('Hourly Purchase Patterns - November 2019',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Key Findings:
- Peak purchasing hours are 8AM - 10AM
- 9AM is the busiest hour with 71K purchases and \$22M revenue
- Sharp drop off after 6PM suggesting working hours drive purchases
- Midnight to 3AM has minimal activity

## 6. Day of Week Analysis
Understanding which days drive the most purchases and revenue.

In [ ]:
df_daily = session.sql("""
    SELECT
        d.DAY_NAME,
        d.DAY_OF_WEEK,
        COUNT(*)                    AS total_purchases,
        ROUND(SUM(f.PRICE), 2)      AS total_revenue,
        ROUND(AVG(f.PRICE), 2)      AS avg_order_value
    FROM FCT_EVENTS f
    JOIN DIM_DATES d
        ON f.EVENT_DATE = d.EVENT_DATE
    WHERE f.IS_PURCHASE = 1
    GROUP BY d.DAY_NAME, d.DAY_OF_WEEK
    ORDER BY d.DAY_OF_WEEK
""").to_pandas()

df_daily

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].bar(df_daily['DAY_NAME'],
            df_daily['TOTAL_PURCHASES'],
            color=['orange' if x in [0,6] 
                   else 'steelblue' 
                   for x in df_daily['DAY_OF_WEEK']])
axes[0].set_title('Purchases by Day of Week')
axes[0].set_xlabel('Day')
axes[0].set_ylabel('Total Purchases')

axes[1].bar(df_daily['DAY_NAME'],
            df_daily['TOTAL_REVENUE'],
            color=['orange' if x in [0,6] 
                   else 'steelblue' 
                   for x in df_daily['DAY_OF_WEEK']])
axes[1].set_title('Revenue by Day of Week')
axes[1].set_xlabel('Day')
axes[1].set_ylabel('Revenue ($)')

plt.suptitle('Day of Week Analysis - November 2019',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Key Findings:
- Sunday is by far the busiest day with 253K purchases (\$77M revenue)
- Weekends account for significantly more purchases than weekdays
- Weekday purchases are very consistent at ~97-105K per day
- AOV is consistent across all days (\$290-\$312) suggesting day doesn't affect spend amount

---
## 7. A/B Testing — Weekend vs Weekday Conversion Rate
Testing whether weekend conversion rates are statistically 
different from weekday conversion rates.

In [ ]:
df_ab = session.sql("""
    SELECT
        d.IS_WEEKEND,
        SUM(f.IS_PURCHASE)                                      AS purchases,
        SUM(f.IS_VIEW)                                          AS views,
        COUNT(DISTINCT f.USER_SESSION)                          AS sessions,
        ROUND(SUM(f.IS_PURCHASE) / SUM(f.IS_VIEW) * 100, 4)    AS conversion_rate
    FROM FCT_EVENTS f
    JOIN DIM_DATES d
        ON f.EVENT_DATE = d.EVENT_DATE
    GROUP BY d.IS_WEEKEND
""").to_pandas()

df_ab

In [ ]:
from scipy import stats

# Weekday data
weekday = df_ab[df_ab['IS_WEEKEND'] == False].iloc[0]
weekend = df_ab[df_ab['IS_WEEKEND'] == True].iloc[0]

# Conversion rates
weekday_rate = weekday['PURCHASES'] / weekday['VIEWS']
weekend_rate = weekend['PURCHASES'] / weekend['VIEWS']

# Z-test for proportions
from statsmodels.stats.proportion import proportions_ztest

counts = [weekend['PURCHASES'], weekday['PURCHASES']]
nobs = [weekend['VIEWS'], weekday['VIEWS']]

stat, pvalue = proportions_ztest(counts, nobs)

print('A/B TEST RESULTS')
print('-' * 40)
print(f"Weekday Conversion Rate : {weekday_rate*100:.4f}%")
print(f"Weekend Conversion Rate : {weekend_rate*100:.4f}%")
print(f"Relative Uplift         : {(weekend_rate/weekday_rate - 1)*100:.2f}%")
print('-' * 40)
print(f"Z-statistic             : {stat:.4f}")
print(f"P-value                 : {pvalue:.10f}")
print('-' * 40)
if pvalue < 0.05:
    print("Result: STATISTICALLY SIGNIFICANT ✅")
    print("Weekend conversion is genuinely higher!")
else:
    print("Result: NOT STATISTICALLY SIGNIFICANT ❌")

### Key Findings:
- Weekend conversion rate (1.80%) is 45% higher than weekday (1.24%)
- P-value is essentially 0 → difference is statistically significant
- With 63M data points this is a genuine behavioural difference
- Business recommendation: increase marketing spend on weekends

---
## 8. Product Performance
Identifying best and worst performing products.

In [ ]:
df_products = session.sql("""
    SELECT
        PRODUCT_ID,
        SUM(IS_VIEW)                                        AS views,
        SUM(IS_CART)                                        AS carts,
        SUM(IS_PURCHASE)                                    AS purchases,
        ROUND(SUM(IS_CART) / NULLIF(SUM(IS_VIEW), 0) * 100, 2)     AS view_to_cart_rate,
        ROUND(SUM(IS_PURCHASE) / NULLIF(SUM(IS_CART), 0) * 100, 2) AS cart_to_purchase_rate,
        ROUND(SUM(IS_PURCHASE) / NULLIF(SUM(IS_VIEW), 0) * 100, 2) AS overall_conversion_rate,
        ROUND(SUM(CASE WHEN IS_PURCHASE = 1 
            THEN PRICE END), 2)                             AS total_revenue
    FROM FCT_EVENTS
    GROUP BY PRODUCT_ID
    HAVING SUM(IS_VIEW) > 100
    ORDER BY overall_conversion_rate DESC
    LIMIT 10
""").to_pandas()

df_products

### Key Findings:
- Top converting products achieve 10-18% conversion rate vs 1.44% average
- Product 12712903 shows 22 purchases with 0 cart adds → possible direct purchase flow
- High conversion products don't always generate highest revenue
- Conversion rate and revenue are different optimization targets

## 9. User Behaviour
Understanding repeat buyers vs one time buyers.

In [ ]:
df_users = session.sql("""
    SELECT
        buyer_type,
        total_users,
        ROUND(total_users * 100.0 / SUM(total_users) OVER(), 2) AS pct_of_users
    FROM (
        SELECT
            buyer_type,
            COUNT(*) AS total_users
        FROM (
            SELECT
                USER_ID,
                CASE 
                    WHEN COUNT(DISTINCT CASE WHEN IS_PURCHASE = 1 
                        THEN USER_SESSION END) = 0 THEN 'No Purchase'
                    WHEN COUNT(DISTINCT CASE WHEN IS_PURCHASE = 1 
                        THEN USER_SESSION END) = 1 THEN 'One Time Buyer'
                    WHEN COUNT(DISTINCT CASE WHEN IS_PURCHASE = 1 
                        THEN USER_SESSION END) = 2 THEN 'Twice Buyer'
                    ELSE 'Repeat Buyer'
                END AS buyer_type
            FROM FCT_EVENTS
            GROUP BY USER_ID
        )
        GROUP BY buyer_type
    )
    ORDER BY total_users DESC
""").to_pandas()

df_users

In [ ]:
colors = ['#d9534f', '#f0ad4e', '#5bc0de', '#5cb85c']

plt.bar(df_users['BUYER_TYPE'],
        df_users['TOTAL_USERS'],
        color=colors)

plt.title('User Breakdown by Purchase Behaviour - November 2019',
          fontweight='bold')
plt.xlabel('Buyer Type')
plt.ylabel('Total Users')
plt.tight_layout()
plt.show()

### Key Findings:
- 88% of users never made a purchase → massive conversion opportunity!
- Only 12% of users ever purchased
- Of buyers, 14% became repeat purchasers
- Recommendation: focus on converting browsers to first time buyers

---
## 10. Executive Summary & Recommendations

### Overall Performance
- 3.7M users generated \$275M revenue in November 2019
- Average order value of \$300 confirms high ticket electronics store
- Overall conversion rate of 1.44% with significant room for improvement

### Key Recommendations

**1. Fix the Funnel**
- View to cart rate is only 4.77% → add quick add to cart, social proof, urgency triggers

**2. Leverage Weekend & Morning Patterns**
- Weekends convert 45% better → increase ad spend and launch products on weekends
- Peak hours 8-10AM → schedule campaigns and notifications for morning window

**3. Convert the 88% Browsers**
- Retargeting campaigns, exit intent popups, first purchase discounts
- Personalized recommendations based on viewed products

**4. Retain Existing Buyers**
- Only 1.68% are repeat buyers → loyalty program and post purchase emails
- Existing buyers are cheaper to retain than acquiring new ones!

**5. Double Down on Electronics & Apple**
- Electronics drives 74.6% of revenue → expand catalogue
- Apple alone generates \$127M → prioritize stock and cross-sell accessories

**6. Recover Abandoned Carts**
- 70% of cart adds never purchase → abandoned cart email reminders
- Even 5% recovery = significant revenue uplift!

**7. Fix Unknown Category**
- \$29M revenue unattributed → work with data team to categorize properly
- Better data at source = better insights!